# z301 — Etapa 1: Preprocesamiento

Input  : `sell-in.txt.gz`, `tb_productos.txt`, `product_id_apredecir201912.txt`
Output : `z301_preprocessed.parquet`

Responsabilidades:
- Completar ceros entre fecha de nacimiento y fecha de muerte de cada producto
- Crear columna `agrupa_id`
- Join con `tb_productos` para agregar jerarquía de categorías
- Switch para usar solo los 780 productos a predecir o todos

## 0. Ambiente

In [ ]:
import os, shutil, subprocess

# El bucket ya está montado por gcsfuse en /home/ds/buckets/b1.
# SQLite de Optuna va al disco LOCAL (/home/ds), no al bucket (gcsfuse no soporta locks).
BASE       = '/home/ds/buckets/b1'
LOCAL_HOME = '/home/ds'

os.makedirs(f'{BASE}/exp',      exist_ok=True)
os.makedirs(f'{BASE}/datasets', exist_ok=True)

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = os.path.expanduser('~/.kaggle/kaggle.json')
os.makedirs(os.path.dirname(kaggle_dst), exist_ok=True)
if os.path.exists(kaggle_dst):
    os.chmod(kaggle_dst, 0o600)
    print('Kaggle auth OK (ya estaba en ~/.kaggle)')
else:
    _encontrado = False
    for cand in [f'{BASE}/kaggle.json', f'{BASE}/kaggle/kaggle.json']:
        if os.path.exists(cand):
            shutil.copy(cand, kaggle_dst)
            os.chmod(kaggle_dst, 0o600)
            print(f'Kaggle auth OK (copiado de {cand})')
            _encontrado = True
            break
    if not _encontrado:
        print('⚠️  kaggle.json no encontrado. Subilo a ~/.kaggle/kaggle.json o al bucket.')

def descargar(archivo):
    url = f'https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}'
    dst = f'{BASE}/datasets/{archivo}'
    if not os.path.exists(dst):
        subprocess.run(['wget', url, '-O', dst], check=True)
    print(f'✅ {archivo}')

descargar('sell-in.txt.gz')
descargar('tb_productos.txt')
descargar('tb_stocks.txt')
descargar('product_id_apredecir201912.txt')

In [ ]:
!pip install -q uv
!uv pip install -q pyarrow fastparquet

## 1. Parámetros

In [ ]:
import os

PARAM = {
    'experimento': 'z301',

    # ── PALANCA 1: granularidad ───────────────────────────────
    # 'producto'         → agrupa_id = product_id
    # 'cliente_producto' → agrupa_id = customer_id * 10000 + product_id
    'modo_agrupacion': 'producto',

    # ── PALANCA 2: universo de productos ─────────────────────────
    'solo_predecir': True,

    # ── Rutas de entrada ──────────────────────────────────
    'path_sellin':    '/home/ds/buckets/b1/datasets/sell-in.txt.gz',
    'path_productos': '/home/ds/buckets/b1/datasets/tb_productos.txt',
    'path_apredecir': '/home/ds/buckets/b1/datasets/product_id_apredecir201912.txt',
}

# El output incluye el modo en el nombre → no se pisan los archivos entre modos
PARAM['path_output'] = f"/home/ds/buckets/b1/exp/z301_preprocessed_{PARAM['modo_agrupacion']}.parquet"

ruta_exp = '/home/ds/buckets/b1/exp/' + PARAM['experimento']
os.makedirs(ruta_exp, exist_ok=True)
os.chdir(ruta_exp)
print('Parámetros:', PARAM)
print('Output:', PARAM['path_output'])

## 2. Carga de datos

In [ ]:
import polars as pl
import numpy as np

df_raw = pl.read_csv(
    PARAM['path_sellin'],
    separator='\t',
    schema_overrides={
        'periodo':           pl.Int32,
        'customer_id':       pl.Int32,
        'product_id':        pl.Int32,
        'cust_request_qty':  pl.Int32,
        'cust_request_tn':   pl.Float32,
        'tn':                pl.Float32,
    }
)
print(f'Sell-in crudo: {df_raw.shape}')

tb_apredecir = pl.read_csv(
    PARAM['path_apredecir'], separator='\t',
    schema_overrides={'product_id': pl.Int32}
)
print(f'Productos a predecir: {tb_apredecir.height}')

tb_productos = pl.read_csv(PARAM['path_productos'], separator='\t')
print(f'Catálogo de productos: {tb_productos.shape}')
print('Columnas:', tb_productos.columns)
tb_productos.head(3)

## 2b. Features de categoría sobre el universo COMPLETO

Se calculan con **todos** los productos (antes de filtrar a los 780), para que los totales de categoría y la competencia incluyan productos fuera de la lista a predecir. Capturan el efecto de canibalización: cuando nace un producto nuevo en una categoría, los demás se ven afectados.

- `tn_cat2_total`: toneladas totales de la categoría por período
- `productos_activos_cat2`: cantidad de productos activos en la categoría
- `productos_nuevos_cat2_3m`: productos nacidos en los últimos 3 meses en la categoría

In [ ]:
# df_raw acá todavía es el sell-in COMPLETO (el filtro a 780 viene después)
prod_cat = tb_productos.select(['product_id', 'cat1', 'cat2', 'cat3', 'brand']).with_columns(
    pl.col('product_id').cast(pl.Int32)
)

# Sell-in completo agregado a producto×período, con su jerarquía
df_univ = (
    df_raw
    .group_by(['product_id', 'periodo'])
    .agg(pl.col('tn').sum().alias('tn'))
    .join(prod_cat, on='product_id', how='left')
)

# Índice ordinal de períodos para ventanas de N meses
periodos_u = sorted(df_univ['periodo'].unique().to_list())
p2i = {p: i for i, p in enumerate(periodos_u)}

# ── 1) Totales jerárquicos por nivel × período (universo completo) ──
# Se calcula el total de toneladas de cada nivel de categoría y marca.
totales = {}
for nivel in ['cat1', 'cat2', 'cat3', 'brand']:
    totales[nivel] = (
        df_univ.group_by([nivel, 'periodo'])
               .agg(pl.col('tn').sum().alias(f'tn_total_{nivel}'))
    )

# ── 2) Productos activos por cat2 y cat3 × período ──
activos = {}
for nivel in ['cat2', 'cat3']:
    activos[nivel] = (
        df_univ.filter(pl.col('tn') > 0)
               .group_by([nivel, 'periodo'])
               .agg(pl.col('product_id').n_unique().alias(f'productos_activos_{nivel}'))
    )

# ── 3) Nacimientos recientes (canibalización) por cat2 y cat3 ──
# nacimiento = primer período con tn>0 de cada producto
nac = (
    df_univ.filter(pl.col('tn') > 0)
           .group_by('product_id')
           .agg(pl.col('periodo').min().alias('nacimiento'))
           .join(prod_cat, on='product_id', how='left')
)

def nacimientos_por_nivel(nivel):
    """Cuenta productos nacidos en la ventana [período-2 ... período] por nivel × período.
    Maneja productos sin categoría asignada (huérfanos del catálogo, ~0.03% del tn,
    ninguno entre los 780 a predecir) agrupándolos bajo el valor null.
    """
    registros = []
    valores = df_univ[nivel].unique().to_list()
    for val in valores:
        if val is None:
            nacs = nac.filter(pl.col(nivel).is_null())['nacimiento'].to_list()
        else:
            nacs = nac.filter(pl.col(nivel) == val)['nacimiento'].to_list()
        for p in periodos_u:
            i = p2i[p]
            ventana = set(periodos_u[max(0, i - 2): i + 1])
            n_nuevos = sum(1 for nn in nacs if nn in ventana)
            registros.append({nivel: val, 'periodo': p,
                              f'productos_nuevos_{nivel}_3m': n_nuevos})
    return pl.DataFrame(registros).with_columns(pl.col('periodo').cast(pl.Int32))

nuevos_cat2 = nacimientos_por_nivel('cat2')
nuevos_cat3 = nacimientos_por_nivel('cat3')

print('Features de categoría (universo completo) calculadas:')
print('  totales:', list(totales.keys()))
print('  activos:', list(activos.keys()))
print('  canibalización: cat2, cat3')

## 3. Filtrado y construcción de AGRUPA_ID

In [ ]:
if PARAM['solo_predecir']:
    df_raw = df_raw.join(tb_apredecir, on='product_id', how='inner')
    print(f'Filtrado a 780 productos: {df_raw.shape}')
else:
    print(f'Usando todos los productos: {df_raw.shape}')

if PARAM['modo_agrupacion'] == 'producto':
    df_agrupado = (
        df_raw
        .group_by(['product_id', 'periodo'])
        .agg(pl.col('tn').sum().alias('tn'))
        .with_columns(
            pl.col('product_id').cast(pl.Int64).alias('agrupa_id'),
            pl.lit(None).cast(pl.Int32).alias('customer_id')
        )
    )
elif PARAM['modo_agrupacion'] == 'cliente_producto':
    # MULTIPLICADOR debe ser > max(product_id) para que la codificación
    # customer_id * MULTIPLICADOR + product_id sea inequívoca (sin colisiones).
    # product_id llega a ~21299 → 100_000 da margen amplio.
    MULTIPLICADOR_AGRUPA = 100_000
    df_agrupado = (
        df_raw
        .group_by(['customer_id', 'product_id', 'periodo'])
        .agg(pl.col('tn').sum().alias('tn'))
        .with_columns(
            (pl.col('customer_id').cast(pl.Int64) * MULTIPLICADOR_AGRUPA
             + pl.col('product_id').cast(pl.Int64)).alias('agrupa_id')
        )
    )
else:
    raise ValueError(f"modo_agrupacion inválido: {PARAM['modo_agrupacion']}")

df_agrupado = df_agrupado.sort(['agrupa_id', 'periodo'])
print(f'Filas agrupadas: {df_agrupado.shape}')
print(f'IDs únicos: {df_agrupado["agrupa_id"].n_unique()}')

## 4. Completar ceros entre nacimiento y muerte de cada producto

Reglas:
- **Nacimiento**: primer período con `tn > 0`
- **Muerte**: último período con `tn > 0`
- Si el último período activo es el último del dataset (201912), NO inferimos que murió
- Ceros se completan solo dentro del rango [nacimiento, muerte], ambos inclusive

In [ ]:
# Todos los períodos del dataset
periodos_all = sorted(df_agrupado['periodo'].unique().to_list())
ULTIMO_PERIODO_DATASET = max(periodos_all)
print(f'Períodos: {min(periodos_all)} → {ULTIMO_PERIODO_DATASET}  ({len(periodos_all)} meses)')

periodo_a_idx = {p: i for i, p in enumerate(periodos_all)}
idx_a_periodo = {i: p for p, i in periodo_a_idx.items()}

# Nacimiento y muerte por agrupa_id
ventas_positivas = df_agrupado.filter(pl.col('tn') > 0)

primeros = (
    ventas_positivas
    .group_by('agrupa_id')
    .agg(pl.col('periodo').min().alias('primer_periodo_activo'))
)

ultimos = (
    ventas_positivas
    .group_by('agrupa_id')
    .agg(pl.col('periodo').max().alias('ultimo_periodo_activo'))
)

primeros_ultimos = primeros.join(ultimos, on='agrupa_id', how='left')
print(f'Productos con historia: {primeros_ultimos.height}')

# Cuántos productos están activos hasta el último período
n_activos = primeros_ultimos.filter(
    pl.col('ultimo_periodo_activo') == ULTIMO_PERIODO_DATASET
).height
print(f'Productos activos en {ULTIMO_PERIODO_DATASET}: {n_activos}')
print(f'Productos discontinuados antes de {ULTIMO_PERIODO_DATASET}: {primeros_ultimos.height - n_activos}')

In [ ]:
# Construir grid: agrupa_id × períodos entre nacimiento y muerte
registros = []

for row in primeros_ultimos.iter_rows(named=True):
    aid     = row['agrupa_id']
    primer_p = row['primer_periodo_activo']
    ultimo_p = row['ultimo_periodo_activo']

    idx_inicio = periodo_a_idx[primer_p]
    idx_fin    = periodo_a_idx[ultimo_p]

    for i in range(idx_inicio, idx_fin + 1):
        registros.append({'agrupa_id': aid, 'periodo': idx_a_periodo[i]})

grid = pl.DataFrame(registros).with_columns([
    pl.col('agrupa_id').cast(pl.Int64),
    pl.col('periodo').cast(pl.Int32),
])
print(f'Grid (agrupa_id × período): {grid.shape}')

# Join con ventas reales; faltantes → 0
df_full = (
    grid
    .join(
        df_agrupado.select(['agrupa_id', 'periodo', 'product_id', 'customer_id', 'tn']),
        on=['agrupa_id', 'periodo'],
        how='left'
    )
    .with_columns(pl.col('tn').fill_null(0.0))
    .sort(['agrupa_id', 'periodo'])
)

print(f'Dataset con ceros completados: {df_full.shape}')
print(f'Nulos en tn: {df_full["tn"].is_null().sum()}')

## 5. Reconstruir product_id en modo producto

In [ ]:
if PARAM['modo_agrupacion'] == 'producto':
    df_full = df_full.with_columns(
        pl.col('agrupa_id').cast(pl.Int32).alias('product_id')
    )
elif PARAM['modo_agrupacion'] == 'cliente_producto':
    # Decodificar customer_id y product_id desde agrupa_id (necesario para las filas
    # de ceros completados, que vienen de la grid y no traen estas columnas).
    MULTIPLICADOR_AGRUPA = 100_000
    df_full = df_full.with_columns([
        (pl.col('agrupa_id') // MULTIPLICADOR_AGRUPA).cast(pl.Int32).alias('customer_id'),
        (pl.col('agrupa_id') % MULTIPLICADOR_AGRUPA).cast(pl.Int32).alias('product_id'),
    ])

print('Schema:', df_full.schema)
print(f'product_id nulos: {df_full["product_id"].is_null().sum()}')
print(f'customer_id nulos: {df_full["customer_id"].is_null().sum()}')

## 6. Join con tb_productos — jerarquía de categorías

Agrega `cat1`, `cat2`, `cat3`, `brand`, `sku_size` como features para LGBM.

In [ ]:
# Asegurar que product_id tiene el mismo tipo en ambas tablas
tb_productos = tb_productos.with_columns(
    pl.col('product_id').cast(pl.Int32)
)

# Ver columnas disponibles
print('Columnas de tb_productos:', tb_productos.columns)

# Join
df_full = df_full.join(tb_productos, on='product_id', how='left')

print(f'Después del join con tb_productos: {df_full.shape}')
print(f'Nulos en cat2: {df_full["cat2"].is_null().sum() if "cat2" in df_full.columns else "cat2 no encontrado"}')
df_full.head(3)

## 6b. Features de estructura de clientes

Calculadas desde el sell-in crudo (tiene `customer_id`) por `product_id × periodo`:
- `clientes_activos`: cantidad de clientes con tn > 0
- `hhi_clientes`: concentración Herfindahl (Σ share²)
- `cliente_principal_share`: share del cliente mayor

Son features del período actual (sin leakage, igual que lag_0).

In [ ]:
# df_raw todavía tiene customer_id (es el sell-in filtrado, antes de agregar por producto)
# Agregar a cliente×producto×período y quedarse con clientes activos
df_cust = (
    df_raw
    .group_by(['product_id', 'customer_id', 'periodo'])
    .agg(pl.col('tn').sum().alias('tn_cust'))
    .filter(pl.col('tn_cust') > 0)
)

# Total por producto×período
df_prod_tot = (
    df_cust
    .group_by(['product_id', 'periodo'])
    .agg(pl.col('tn_cust').sum().alias('tn_prod_tot'))
)

# 1) Cantidad de clientes activos
df_n_cust = (
    df_cust
    .group_by(['product_id', 'periodo'])
    .agg(pl.col('customer_id').n_unique().alias('clientes_activos'))
)

# 2) y 3) Concentración (HHI) y cliente principal
df_cust = df_cust.join(df_prod_tot, on=['product_id', 'periodo'], how='left')
df_cust = df_cust.with_columns(
    (pl.col('tn_cust') / pl.col('tn_prod_tot')).alias('share')
)
df_conc = (
    df_cust
    .group_by(['product_id', 'periodo'])
    .agg([
        pl.col('share').max().alias('cliente_principal_share'),
        (pl.col('share') ** 2).sum().alias('hhi_clientes'),
    ])
)

# Unir las 3 features
df_cust_feats = df_n_cust.join(df_conc, on=['product_id', 'periodo'], how='left')
print(f'Features de clientes calculadas: {df_cust_feats.shape}')

# Cast tipos
df_cust_feats = df_cust_feats.with_columns([
    pl.col('clientes_activos').cast(pl.Int32),
    pl.col('cliente_principal_share').cast(pl.Float32),
    pl.col('hhi_clientes').cast(pl.Float32),
])

# Join al dataset principal por product_id × periodo
# (en modo cliente_producto agrega las stats del producto a cada fila de cliente)
df_full = df_full.join(df_cust_feats, on=['product_id', 'periodo'], how='left')

# Los períodos completados con cero (sin ventas) quedan null → rellenar
df_full = df_full.with_columns([
    pl.col('clientes_activos').fill_null(0),
    pl.col('cliente_principal_share').fill_null(0.0),
    pl.col('hhi_clientes').fill_null(0.0),
])

print('Producto 20001 (muestra):')
print(df_full.filter(pl.col('product_id') == 20001).select(
    ['periodo', 'tn', 'clientes_activos', 'cliente_principal_share', 'hhi_clientes']
).head(5))

## 6c. Unir features de categoría y calcular market share

Se unen las features de categoría (calculadas sobre el universo completo) y se calcula `market_share = tn / tn_cat2_total` con el total correcto.

In [ ]:
# Unir todos los totales jerárquicos (cada uno por su nivel × periodo)
for nivel in ['cat1', 'cat2', 'cat3', 'brand']:
    df_full = df_full.join(totales[nivel], on=[nivel, 'periodo'], how='left')

# Unir productos activos (cat2, cat3)
for nivel in ['cat2', 'cat3']:
    df_full = df_full.join(activos[nivel], on=[nivel, 'periodo'], how='left')

# Unir canibalización (cat2, cat3)
df_full = df_full.join(nuevos_cat2, on=['cat2', 'periodo'], how='left')
df_full = df_full.join(nuevos_cat3, on=['cat3', 'periodo'], how='left')

# Rellenar nulos de las features de categoría
df_full = df_full.with_columns([
    pl.col('tn_total_cat1').fill_null(0.0),
    pl.col('tn_total_cat2').fill_null(0.0),
    pl.col('tn_total_cat3').fill_null(0.0),
    pl.col('tn_total_brand').fill_null(0.0),
    pl.col('productos_activos_cat2').fill_null(0),
    pl.col('productos_activos_cat3').fill_null(0),
    pl.col('productos_nuevos_cat2_3m').fill_null(0),
    pl.col('productos_nuevos_cat3_3m').fill_null(0),
])

# Market share a varios niveles (tn del producto / total del nivel)
# Nota: en modo producto tn es la del producto; en cliente_producto es la del cliente-producto
df_full = df_full.with_columns([
    pl.when(pl.col('tn_total_cat2') > 0).then(pl.col('tn') / pl.col('tn_total_cat2'))
      .otherwise(pl.lit(0.0)).alias('market_share_cat2'),
    pl.when(pl.col('tn_total_cat3') > 0).then(pl.col('tn') / pl.col('tn_total_cat3'))
      .otherwise(pl.lit(0.0)).alias('market_share_cat3'),
    pl.when(pl.col('tn_total_brand') > 0).then(pl.col('tn') / pl.col('tn_total_brand'))
      .otherwise(pl.lit(0.0)).alias('market_share_brand'),
])
# Mantener 'market_share' (alias del de cat2) por compatibilidad con experimentos previos
df_full = df_full.with_columns(pl.col('market_share_cat2').alias('market_share'))

print('Features jerárquicas unidas. Muestra producto 20001:')
print(df_full.filter(pl.col('product_id') == 20001).select(
    ['periodo', 'tn', 'tn_total_cat2', 'tn_total_cat3', 'market_share_cat2',
     'market_share_cat3', 'productos_nuevos_cat2_3m', 'productos_nuevos_cat3_3m']
).head(5))

## 7. Metadatos y verificación

In [ ]:
df_full = df_full.with_columns([
    pl.lit(PARAM['modo_agrupacion']).alias('modo_agrupacion'),
    pl.lit(PARAM['solo_predecir']).alias('solo_predecir'),
])

# Verificación producto estrella
print('Producto 20001:')
print(df_full.filter(pl.col('product_id') == 20001).select(
    ['product_id', 'periodo', 'tn', 'cat1', 'cat2', 'brand']
).head(5))

## 8. Guardar output

In [ ]:
df_full.write_parquet(PARAM['path_output'])
print(f'✅ Guardado: {PARAM["path_output"]}')
print(f'   Filas: {df_full.height:,}')
print(f'   Columnas: {df_full.columns}')

# Verificación
df_check = pl.read_parquet(PARAM['path_output'])
print(f'   Lectura OK: {df_check.shape}')